<a href="https://colab.research.google.com/github/sihussain2/grpc/blob/master/IntradayTrading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance pandas numpy

In [1]:
#!/usr/bin/env python3
"""
Intraday Research Script for Google Colab

- Picks 3 tickers per run from a default list
- Fetches daily and intraday data via yfinance
- Computes indicators: SMA, EMA, RSI, ATR, Bollinger Bands, volume spikes
- Scores BUY/HOLD/SELL
- Runs 3-month backtest for 7 and 10-day holding
- Sends HTML table report to Gmail
- Saves CSV report
"""

# ---------------------
# Install dependencies (run once in Colab)
# ---------------------


# ---------------------
# IMPORTS
# ---------------------
import yfinance as yf
import pandas as pd
import numpy as np
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta, UTC
import random

# ---------------------
# CONFIGURATION
# ---------------------
TICKERS_PER_RUN = 3
DAILY_LOOKBACK_DAYS = 90  # 3 months backtest
DEFAULT_POSITION_PCT = 0.02
ATR_MULTIPLIER_SL = 1.5

# Gmail SMTP config
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587
SMTP_USER = "sihussain2@gmail.com"
SMTP_PASSWORD = "dhdicjiawlxsweyr"  # App password
EMAIL_FROM = SMTP_USER
EMAIL_TO = ["sihussain2@gmail.com"]
EMAIL_SUBJECT = "Intraday Research - Daily Report"

# Default company list - using ticker symbols directly
DEFAULT_TICKERS = [
    "AAPL", "MSFT", "AMZN", "NVDA", "TSLA", "META", "GOOG", "JPM",
    "WFC", "BAC", "SHOP", "RY", "TD", "BNS", "ENB", "BKR",
    "CVX", "XOM", "PFE", "MRK"
]

# ---------------------
# HELPER FUNCTIONS
# ---------------------

def add_indicators(df, intraday=False):
    df = df.copy()
    df['SMA20'] = df['Close'].rolling(20, min_periods=1).mean()
    df['SMA50'] = df['Close'].rolling(50, min_periods=1).mean()
    df['EMA9'] = df['Close'].ewm(span=9, adjust=False).mean()
    # RSI
    delta = df['Close'].diff()
    up = delta.clip(lower=0)
    down = -1 * delta.clip(upper=0)
    ma_up = up.rolling(14, min_periods=1).mean()
    ma_down = down.rolling(14, min_periods=1).mean()
    rs = ma_up / ma_down.replace(0, np.nan)
    df['RSI14'] = 100 - (100 / (1 + rs))
    # ATR
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(14, min_periods=1).mean()
    # Bollinger Bands
    df['BB_mid'] = df['Close'].rolling(20, min_periods=1).mean()
    df['BB_std'] = df['Close'].rolling(20, min_periods=1).std().fillna(0)
    df['BB_upper'] = df['BB_mid'] + 2 * df['BB_std']
    df['BB_lower'] = df['BB_mid'] - 2 * df['BB_std']
    # VWAP for intraday
    if intraday:
        tp = (df['High'] + df['Low'] + df['Close']) / 3
        df['vwap'] = (tp * df['Volume']).cumsum() / df['Volume'].cumsum()
    else:
        df['vwap'] = np.nan
    # volume spike
    df['vol_mean_20'] = df['Volume'].rolling(20, min_periods=1).mean()
    df['vol_spike'] = df['Volume'] > (df['vol_mean_20'] * 2.5)
    return df

def score_and_signal(latest, fundamentals):
    score = 50
    reasons = []
    if latest['Close'] > latest['SMA20']:
        score += 10; reasons.append("Price > SMA20")
    else: score -= 8; reasons.append("Price < SMA20")
    if latest['SMA20'] > latest['SMA50']:
        score += 8; reasons.append("SMA20 > SMA50")
    else: score -=6; reasons.append("SMA20 < SMA50")
    if latest['RSI14'] < 30: score +=5; reasons.append("Oversold RSI")
    elif latest['RSI14'] > 70: score -=7; reasons.append("Overbought RSI")
    if pd.notna(latest['ATR14']) and latest['ATR14']/latest['Close']>0.05: score-=5; reasons.append("High volatility (ATR)")
    if latest.get('vol_spike', False):
        if latest['Close'] > latest['Open']: score +=6; reasons.append("Volume spike up candle")
        else: score -=4; reasons.append("Volume spike down candle")
    mcap = fundamentals.get('marketCap',0)
    if mcap and mcap<300_000_000: score-=10; reasons.append("Small market cap")
    score = max(0,min(100,score))
    if score>=65: rec="BUY"
    elif score>=45: rec="HOLD"
    else: rec="SELL"
    return {"score":int(score),"rec":rec,"reasons":reasons}

def compute_rsi(series, period=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -1*delta.clip(upper=0)
    ma_up = up.rolling(period,min_periods=1).mean()
    ma_down = down.rolling(period,min_periods=1).mean()
    rs = ma_up / ma_down.replace(0,np.nan)
    return 100 - (100/(1+rs))

def compute_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period,min_periods=1).mean()

def backtest_simple(ticker, daily_df, rule_params):
    df = daily_df.copy().dropna().reset_index(drop=False).rename(columns={'index':'Date'})
    df['SMA20'] = df['Close'].rolling(20,min_periods=1).mean()
    df['SMA50'] = df['Close'].rolling(50,min_periods=1).mean()
    df['RSI14'] = compute_rsi(df['Close'],14)
    df['ATR14'] = compute_atr(df,14)
    results=[]
    for i in range(1,len(df)-1):
        row = df.loc[i]
        if (row['Close']>row['SMA20']) and (row['SMA20']>row['SMA50']) and (row['RSI14']<70):
            entry_index=i+1
            if entry_index>=len(df): break
            entry_price=df.at[entry_index,'Open']
            atr = df.at[entry_index,'ATR14'] if pd.notna(df.at[entry_index,'ATR14']) else 0
            stop_price=entry_price-(atr*rule_params['atr_mult'])
            max_hold=rule_params['max_days']
            exit_price=None
            exit_date=None
            for hold_i in range(entry_index,min(entry_index+max_hold,len(df))):
                low=df.at[hold_i,'Low']
                if (atr>0) and (low<=stop_price):
                    exit_price=stop_price
                    exit_date=df.at[hold_i,'Date']
                    break
            if exit_price is None:
                last_idx=min(entry_index+max_hold-1,len(df)-1)
                exit_price=df.at[last_idx,'Close']
                exit_date=df.at[last_idx,'Date']
            ret=(exit_price-entry_price)/entry_price
            results.append(ret)
    if results:
        wins=[r for r in results if r>0]
        loss=[r for r in results if r<=0]
        stats={"trades":len(results),"win_rate":len(wins)/len(results),"avg_return":np.mean(results),
               "avg_win":np.mean(wins) if wins else 0.0,"avg_loss":np.mean(loss) if loss else 0.0,
               "max_drawdown":float(np.min(results))}
    else:
        stats={"trades":0,"win_rate":0.0,"avg_return":0.0,"avg_win":0.0,"avg_loss":0.0,"max_drawdown":0.0}
    return stats

def process_ticker(ticker):
    out={"ticker":ticker,"success":False}
    try:
        t=yf.Ticker(ticker)
        info=t.info if hasattr(t,"info") else {}
        fundamentals={"marketCap":info.get("marketCap",0),"trailingPE":info.get("trailingPE"),
                      "shortRatio":info.get("shortRatio"),"averageVolume":info.get("averageVolume")}
        # Daily data
        start=(datetime.now(UTC)-timedelta(days=DAILY_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
        daily=t.history(start=start, interval="1d", auto_adjust=False, actions=False)
        if daily.empty:
            out.update({"success":False,"reason":"No daily data"}); return out
        daily_idx=add_indicators(daily.copy(),intraday=False)
        latest=daily_idx.iloc[-1]
        score_data=score_and_signal(latest,fundamentals)
        atr=latest.get('ATR14',np.nan)
        close=latest.get('Close',np.nan)
        stop_loss=close-(ATR_MULTIPLIER_SL*atr) if pd.notna(atr) and pd.notna(close) and atr>0 else None
        suggested_position_pct=DEFAULT_POSITION_PCT
        bt_7=backtest_simple(ticker,daily,{"atr_mult":ATR_MULTIPLIER_SL,"max_days":7})
        bt_10=backtest_simple(ticker,daily,{"atr_mult":ATR_MULTIPLIER_SL,"max_days":10})
        out.update({"success":True,"fundamentals":fundamentals,"score_data":score_data,
                    "stop_loss":stop_loss,"position_pct":suggested_position_pct,
                    "backtest_7d":bt_7,"backtest_10d":bt_10})
    except Exception as e:
        out.update({"success":False,"reason":str(e)})
    return out

# ---------------------
# MAIN LOGIC
# ---------------------
def main():
    tickers=random.sample(DEFAULT_TICKERS,TICKERS_PER_RUN)
    results=[]
    for tk in tickers:
        res=process_ticker(tk)
        results.append(res)
    # Build DataFrame
    flat_rows=[]
    for r in results:
        if not r.get("success"): continue
        row={
            "Ticker":r["ticker"],
            "Score":r["score_data"]["score"],
            "Rec":r["score_data"]["rec"],
            "Reasons":"; ".join(r["score_data"]["reasons"]),
            "StopLoss":r["stop_loss"],
            "PositionPct":r["position_pct"],
            "Backtest7DTrades":r["backtest_7d"]["trades"],
            "Backtest7DWinRate":r["backtest_7d"]["win_rate"],
            "Backtest10DTrades":r["backtest_10d"]["trades"],
            "Backtest10DWinRate":r["backtest_10d"]["win_rate"]
        }
        flat_rows.append(row)
    df_out=pd.DataFrame(flat_rows)
    # Save CSV
    timestamp=datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    csv_path=f"report_{timestamp}.csv"
    df_out.to_csv(csv_path,index=False)
    print(f"CSV saved: {csv_path}")
    # Send Email
    try:
        msg=MIMEMultipart("alternative")
        msg["Subject"]=EMAIL_SUBJECT
        msg["From"]=EMAIL_FROM
        msg["To"]=", ".join(EMAIL_TO)
        html=df_out.to_html(index=False)
        msg.attach(MIMEText(html,"html"))
        server=smtplib.SMTP(SMTP_HOST,SMTP_PORT)
        server.starttls()
        server.login(SMTP_USER,SMTP_PASSWORD)
        server.sendmail(EMAIL_FROM,EMAIL_TO,msg.as_string())
        server.quit()
        print(f"Email sent to {EMAIL_TO}")
    except Exception as e:
        print(f"Email sending failed: {e}")

if __name__=="__main__":
    main()

CSV saved: report_20251018T132753Z.csv
Email sent to ['sihussain2@gmail.com']


# New Section